# A two-layer classifier as an nn.Module

Write the standard PyTorch classifier: a module with two linear layers and a ReLU, loaded with weights that came from somewhere else.

Write `mlp_predict(x, W1, b1, W2, b2)`:

- `x`: float64 tensor `(N, D)`.
- `W1`: `(D, H)`, `b1`: `(H,)`, `W2`: `(H, C)`, `b2`: `(C,)`, all float64. The weights are given in **math layout**: the layer computes $x W_1 + b_1$.
- Build an `nn.Module` (a subclass or `nn.Sequential`) made of `nn.Linear(D, H)`, `nn.ReLU()`, `nn.Linear(H, C)`, in float64, and copy these weights into it.

Return a tuple `(logits, pred, n_params)`:

- `logits`: `(N, C)` float64, the module's output.
- `pred`: `(N,)` int64, the predicted class per row. Ties go to the smallest class index.
- `n_params`: int, the total number of trainable scalars in the module.

Compute the output in inference mode (no autograd graph).

This problem runs in **Colab** (PyTorch isn't available in the browser). The notebook includes the same tests.

Write your solution in the next cell, then run the test cell.

In [ ]:
import torch
from torch import nn

def mlp_predict(x, W1, b1, W2, b2):
    """Return (logits (N, C), pred (N,), n_params)."""
    D, H = W1.shape
    C = W2.shape[1]
    model = nn.Sequential(nn.Linear(D, H), nn.ReLU(), nn.Linear(H, C)).double()
    with torch.no_grad():
        logits = model(x)
    return logits, logits.argmax(dim=1), 0


In [ ]:
# Tests (same harness and cases as the website)
_HARNESS = '"""Test harness shared by build.py (CPython) and the browser runner (Pyodide).\n\nOne file, so a test that passes at build time is graded the same way in the browser.\nValues cross JSON with small tags: ndarray, tuple, DataFrame/Series, non-finite floats.\n"""\nimport json, math\n\ntry:\n    import numpy as np\nexcept ImportError:  # pure-python problems never need it\n    np = None\n\n\ndef enc(x):\n    if np is not None and isinstance(x, np.ndarray):\n        return {\'__nd__\': enc(x.tolist()), \'dtype\': str(x.dtype)}\n    if np is not None and isinstance(x, np.generic):\n        return enc(x.item())\n    if isinstance(x, float) and not math.isfinite(x):\n        return {\'__f__\': repr(x)}\n    if isinstance(x, tuple):\n        return {\'__tup__\': [enc(v) for v in x]}\n    if isinstance(x, list):\n        return [enc(v) for v in x]\n    if isinstance(x, dict):\n        return {\'__dict__\': [[enc(k), enc(v)] for k, v in x.items()]}\n    t = type(x).__name__\n    if t == \'Tensor\':  # torch, in Colab and at build time\n        return enc(x.detach().cpu().numpy())\n    if t == \'DataFrame\':\n        # Cells go through enc() as Python values, so floats keep full precision (to_json caps at 15 digits).\n        return {\'__df__\': {\'columns\': [enc(c) for c in x.columns], \'data\': [enc(list(r)) for r in x.itertuples(index=False, name=None)]},\n                \'dtypes\': [str(d) for d in x.dtypes]}\n    if t == \'Series\':\n        return {\'__sr__\': enc(x.tolist()), \'name\': x.name}\n    return x\n\n\ndef dec(x):\n    if isinstance(x, list):\n        return [dec(v) for v in x]\n    if not isinstance(x, dict):\n        return x\n    if \'__nd__\' in x:\n        return np.array(dec(x[\'__nd__\']), dtype=x[\'dtype\'])\n    if \'__f__\' in x:\n        return float(x[\'__f__\'])\n    if \'__tup__\' in x:\n        return tuple(dec(v) for v in x[\'__tup__\'])\n    if \'__dict__\' in x:\n        return {_hashable(dec(k)): dec(v) for k, v in x[\'__dict__\']}\n    if \'__df__\' in x:\n        import pandas as pd\n        s = x[\'__df__\']\n        df = pd.DataFrame([dec(r) for r in s[\'data\']], columns=[dec(c) for c in s[\'columns\']])\n        for c, d in zip(df.columns, x[\'dtypes\']):\n            if d.startswith((\'int\', \'float\', \'bool\')):\n                df[c] = df[c].astype(d)\n        return df\n    if \'__sr__\' in x:\n        import pandas as pd\n        return pd.Series(dec(x[\'__sr__\']), name=x[\'name\'])\n    return x\n\n\ndef _hashable(k):\n    return tuple(k) if isinstance(k, list) else k\n\n\ndef same(got, exp, rtol=1e-6, atol=1e-8, path=\'output\'):\n    """(ok, why). Floats compare with tolerance, everything else exactly. Shapes must match."""\n    if type(exp).__name__ == \'DataFrame\':\n        if type(got).__name__ != \'DataFrame\':\n            return False, f\'{path}: expected a DataFrame, got {type(got).__name__}\'\n        if list(got.columns) != list(exp.columns):\n            return False, f\'{path}: columns {list(got.columns)} != expected {list(exp.columns)}\'\n        if got.shape != exp.shape:\n            return False, f\'{path}: shape {got.shape} != expected {exp.shape}\'\n        for c in exp.columns:\n            ok, why = same(got[c].tolist(), exp[c].tolist(), rtol, atol, f\'{path}[{c!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if type(exp).__name__ == \'Series\':\n        exp = exp.tolist()\n        got = got.tolist() if type(got).__name__ == \'Series\' else got\n    if np is not None and isinstance(exp, np.ndarray):\n        try:\n            g = np.asarray(got)\n        except Exception:\n            return False, f\'{path}: expected an array, got {type(got).__name__}\'\n        if g.shape != exp.shape:\n            return False, f\'{path}: shape {g.shape} != expected {exp.shape}\'\n        if exp.dtype.kind in \'fc\' or g.dtype.kind in \'fc\':\n            if g.dtype.kind not in \'fciub\':\n                return False, f\'{path}: expected numbers, got dtype {g.dtype}\'\n            bad = ~np.isclose(g.astype(complex if exp.dtype.kind == \'c\' else float), exp, rtol=rtol, atol=atol, equal_nan=True)\n            if bad.any():\n                i = tuple(int(v) for v in np.argwhere(bad)[0])\n                return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n            return True, \'\'\n        if not np.array_equal(g, exp):\n            i = tuple(int(v) for v in np.argwhere(g != exp)[0]) if g.shape else ()\n            return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n        return True, \'\'\n    if isinstance(exp, float) or (isinstance(exp, int) and isinstance(got, float) and not isinstance(exp, bool)):\n        if np is not None and isinstance(got, np.generic):\n            got = got.item()\n        if not isinstance(got, (int, float)) or isinstance(got, bool):\n            return False, f\'{path}: expected a number, got {type(got).__name__}\'\n        if math.isnan(exp) and math.isnan(got):\n            return True, \'\'\n        return (True, \'\') if math.isclose(got, exp, rel_tol=rtol, abs_tol=atol) else (False, f\'{path}: {got!r} != expected {exp!r}\')\n    if isinstance(exp, (list, tuple)):\n        if np is not None and isinstance(got, np.ndarray):\n            got = got.tolist()\n        if not isinstance(got, (list, tuple)):\n            return False, f\'{path}: expected a {type(exp).__name__}, got {type(got).__name__}\'\n        if len(got) != len(exp):\n            return False, f\'{path}: length {len(got)} != expected {len(exp)}\'\n        for i, (g, e) in enumerate(zip(got, exp)):\n            ok, why = same(g, e, rtol, atol, f\'{path}[{i}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if isinstance(exp, dict):\n        if not isinstance(got, dict):\n            return False, f\'{path}: expected a dict, got {type(got).__name__}\'\n        if set(got) != set(exp):\n            return False, f\'{path}: keys {sorted(map(str, got))} != expected {sorted(map(str, exp))}\'\n        for k in exp:\n            ok, why = same(got[k], exp[k], rtol, atol, f\'{path}[{k!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if np is not None and isinstance(got, np.generic):\n        got = got.item()\n    return (True, \'\') if got == exp else (False, f\'{path}: {got!r} != expected {exp!r}\')\n\n\ndef run_sql(query, setup):\n    """Run `setup` (CREATE/INSERT script) in a fresh in-memory db, then `query`; rows as lists."""\n    import sqlite3  # a separate package in Pyodide: only SQL problems load it\n    con = sqlite3.connect(\':memory:\')\n    try:\n        con.executescript(setup)\n        cur = con.execute(query)\n        return [[c[0] for c in cur.description], [list(r) for r in cur.fetchall()]]\n    finally:\n        con.close()\n\n\ndef show_args(args, runtime, n=600):\n    """Arguments as the learner would write them: one per line, SQL setup as-is."""\n    if runtime == \'sql\':\n        return args[0] if len(args[0]) <= n else args[0][:n] + \' …\'\n    return \'\\n\'.join(short(a, n) for a in args)\n\n\ndef short(x, n=300):\n    s = repr(x)\n    return s if len(s) <= n else s[:n] + \' …\'\n\n\ndef grade(fn, tests, runtime, rtol=1e-6, atol=1e-8):\n    """Run `fn` (a callable, or SQL text for runtime \'sql\') on each test; list of result dicts."""\n    out = []\n    for t in tests:\n        args, exp = prep(dec(t[\'a\']), runtime), dec(t[\'e\'])\n        try:\n            got = run_sql(fn, args[0]) if runtime == \'sql\' else plain(fn(*_copy(args)))\n            ok, why = same(got, exp, rtol, atol)\n        except Exception as e:  # learner code raised: report it like a failed test\n            got, ok, why = None, False, f\'{type(e).__name__}: {e}\'\n        out.append({\'ok\': ok, \'why\': why, \'got\': short(got) if not ok else \'\',\n                    \'inp\': show_args(args, runtime, 400) if not ok else \'\'})\n    return out\n\n\ndef prep(args, runtime):\n    """Arrays cross JSON as ndarrays; torch problems take tensors."""\n    if runtime == \'torch\':\n        import torch\n        return [torch.as_tensor(a) if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    if runtime == \'triton\':                                  # Colab only: kernels need CUDA tensors\n        import torch\n        return [torch.as_tensor(a).cuda() if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    return args\n\n\ndef plain(x):\n    """Framework arrays (torch, jax) -> numpy, recursively, so results compare and encode the same way."""\n    t = type(x).__name__\n    if t == \'Tensor\':\n        return x.detach().cpu().numpy()\n    if t in (\'ArrayImpl\', \'Array\') and hasattr(x, \'__array__\'):\n        return np.asarray(x)\n    if isinstance(x, tuple):\n        return tuple(plain(v) for v in x)\n    if isinstance(x, list):\n        return [plain(v) for v in x]\n    if isinstance(x, dict):\n        return {k: plain(v) for k, v in x.items()}\n    return x\n\n\ndef _copy(args):\n    # Tests reuse inputs across Run and Submit, so a function that mutates its input mustn\'t poison later cases.\n    import copy\n    return copy.deepcopy(args)\n'
_TESTS = '[{"a": [{"__nd__": [[1.0, 2.0]], "dtype": "float64"}, {"__nd__": [[1.0, -1.0], [0.0, 1.0]], "dtype": "float64"}, {"__nd__": [0.0, 0.0], "dtype": "float64"}, {"__nd__": [[1.0, 0.0], [0.0, 1.0]], "dtype": "float64"}, {"__nd__": [0.0, 0.5], "dtype": "float64"}], "e": {"__tup__": [{"__nd__": [[1.0, 1.5]], "dtype": "float64"}, {"__nd__": [1], "dtype": "int64"}, 12]}}, {"a": [{"__nd__": [[1.0], [-1.0]], "dtype": "float64"}, {"__nd__": [[2.0, -3.0, 0.5]], "dtype": "float64"}, {"__nd__": [0.0, 1.0, 0.0], "dtype": "float64"}, {"__nd__": [[1.0], [1.0], [-2.0]], "dtype": "float64"}, {"__nd__": [0.1], "dtype": "float64"}], "e": {"__tup__": [{"__nd__": [[1.1], [4.1]], "dtype": "float64"}, {"__nd__": [0, 0], "dtype": "int64"}, 10]}}, {"a": [{"__nd__": [[-0.8019314252534474, -1.324358995628145, -0.24836162209524854, 0.4204452380655215], [1.1360465324896427, 0.10970639932180819, -0.5526473205362324, -0.7847803553442784], [0.7487457707345911, 1.6347830429585775, 0.27276877584472176, -1.2333286640307717], [-0.9582652054360887, 1.6000190889991115, 0.2028824405086084, -1.7321348424395848], [-0.08369619281702581, -1.1632259734447485, -0.6292880940615545, -0.48800582327685743]], "dtype": "float64"}, {"__nd__": [[-0.7133133716322436, 0.5533784703532895, -0.06308597192528916, -0.5894312580326048, 0.40963782655711695, 0.8298553070613239], [-1.643023371405677, -0.256730126365494, -0.9807473560440125, -0.17315522486203205, -1.2894187467538587, 0.0206903940375912], [-0.03788574104406823, -0.304337750958489, -1.0479265051202462, -0.3961903304730927, -1.091328901695709, -1.3552087462047395], [0.22478573245989314, -1.109349937891366, 1.1702961011782933, 0.7165876558738361, -1.9978166924497212, 0.272128869412488]], "dtype": "float64"}, {"__nd__": [-1.1017166275810448, 0.033057220158269195, 0.04363199256942161, -1.9884297882311208, -0.23342252376577002, -0.255790031399391], "dtype": "float64"}, {"__nd__": [[0.9620005318430944, -1.1814468079562157, 0.7380418978456841], [-1.0989727630364063, -0.33129089269991674, -0.8404731684222111], [1.448731288921672, 0.5682130997882933, 2.4317325028452124], [0.6419163790823205, 0.8449927337191754, 0.8406828762653401], [-0.6066115359095516, -0.07002844663838208, 1.350388867744626], [-0.3965507651729716, 0.18879953129109867, -0.021223460417289796]], "dtype": "float64"}, {"__nd__": [0.6092164928327407, -0.3649087419473726, -0.15236188875684828], "dtype": "float64"}], "e": {"__tup__": [{"__nd__": [[5.051100199695952, -1.254009371546738, 7.135278129118211], [-3.086094100002555, -0.846098414050783, 1.4699310568667845], [-0.9134651178103185, -0.8090730173343441, -1.0780135714267591], [-0.7704324638226651, -0.7187481866397996, -0.20958630523107036], [0.19287188487883433, -1.0341647190688004, 6.577899662279744]], "dtype": "float64"}, {"__nd__": [2, 2, 1, 2, 2], "dtype": "int64"}, 51]}}, {"a": [{"__nd__": [[0.24238142867215043, 0.10302313848680769, -0.8649727462818201]], "dtype": "float64"}, {"__nd__": [[0.08957830431894438, -0.1298481208246912], [-0.12011155488035266, -0.1282491794740462], [0.09669722789332148, -0.036060836889927]], "dtype": "float64"}, {"__nd__": [-0.09710363785210656, -0.11360213941896467], "dtype": "float64"}, {"__nd__": [[0.04211311374624062, -0.1054840662577835, -0.12720782100976422, 0.06139930624688609], [-0.11967077271925707, -0.03224381507513973, -0.0006761540346521786, -0.04453353669298123]], "dtype": "float64"}, {"__nd__": [-0.005409396159544828, 0.13387734990547714, -0.0516894152589275, -0.12593071588782276], "dtype": "float64"}], "e": {"__tup__": [{"__nd__": [[-0.005409396159544828, 0.13387734990547714, -0.0516894152589275, -0.12593071588782276]], "dtype": "float64"}, {"__nd__": [1], "dtype": "int64"}, 20]}}, {"a": [{"__nd__": [[0.0, 0.0], [0.0, 0.0], [0.0, 0.0]], "dtype": "float64"}, {"__nd__": [[1.0, 1.0], [1.0, 1.0]], "dtype": "float64"}, {"__nd__": [-1.0, -1.0], "dtype": "float64"}, {"__nd__": [[1.0, 1.0, 1.0], [1.0, 1.0, 1.0]], "dtype": "float64"}, {"__nd__": [0.0, 0.0, 0.0], "dtype": "float64"}], "e": {"__tup__": [{"__nd__": [[0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0]], "dtype": "float64"}, {"__nd__": [0, 0, 0], "dtype": "int64"}, 15]}}, {"a": [{"__nd__": [[-1.8367457051936795, -0.20476611683386706, -0.3522570104112305, 0.2650909069303235, -0.4642445916374816, -0.4786384630527245, -0.7213157478486046, -0.5197597723701788], [0.1602267063174243, -0.3803527272151388, 0.1004418528325065, 1.9011872000332568, 0.4791181507969435, -1.5760898454352261, 1.7335295856131068, 0.34781036998853543], [-0.9414132864509847, 0.9070489576858102, 0.01765561090983897, -0.6152185330350691, -0.6335089254336164, -0.9934317902518573, 0.048118983989838576, 1.0688166937495658], [-0.3250520988063787, 0.42082412505635486, 1.875646053775279, -1.2145906329798497, 0.2575223239234917, -0.30643692685187435, -1.0592560998278446, -1.0258471994365972], [-0.0152851431653622, 0.4338634112001693, -0.531589820743032, 0.05439054435949247, 0.7222086544347099, 0.2965612469178253, 0.8686701897408925, 0.4613956081509272], [0.4880056756807095, 1.8266340649418156, 0.6231718736925668, 0.12947113235560923, 0.7099438034419987, -0.9191236755082103, -0.3377047857608698, 0.793220854194603], [0.6307867000938839, 1.5483616908312823, 0.010473028918682835, -1.4623277904204515, 1.9472469308762064, 1.0928928105587499, -1.0587374336035666, 1.3758236699684163], [0.03240279877908362, -1.813928769904867, -0.4203539214229468, -0.5089038339937678, 1.590848160322242, -0.7920686954195238, -0.25361734037936806, -0.2767608068453714], [-0.3789727629772628, -0.9140740610326185, 0.21908690902567676, 1.0768046464135546, 0.6239776511717521, -0.9274991160593102, -1.1498157571616898, 0.11897355358204866], [-0.7065801636237695, -0.6301656488493008, -1.6796945788401643, 1.9504916026999646, 0.9166191525408893, -0.9739073319069764, 0.9082102667945468, 1.3424871058246228], [-2.389553349676638, -0.548944785564265, -0.3879732181256381, 0.6482544703432906, -0.12146542230081914, -0.2304313358523204, -0.05837138907215449, 1.8533257078420022], [2.159980469779012, -0.5248264370136562, -0.9262439932504336, 2.6925531473868567, -0.9797446331927372, -0.5734020178922011, 0.036581509348738155, 0.4832462441452338]], "dtype": "float64"}, {"__nd__": [[1.029035549437811, 0.39097722167013954, -0.8723830716951919, 0.5063823263497103, 0.24991757985122218, 1.876918992454176, -0.01499979815571943, -1.3369728999172332, -1.0450199143839616, 1.450153058953162, -0.5401312070783946, -2.1044660100855603, -0.5806997054894565, 1.5099831293121058e-05, 1.1888306785373703, -1.014468137602427], [0.6666833259020761, 0.7952990996016167, -0.6993883083236738, -0.18758970531896946, 1.7694502363979239, 1.720484746826155, 0.8555220049018919, 0.33194635951240653, 1.1383096209632015, -0.14067728561527593, -0.09509306569845781, -0.8603601395273159, 0.005563384668739004, -0.0820166249133826, 2.7736316673693673, -0.19284316578043667], [1.2708179900134384, 1.3204972743933365, -0.18647870191497867, 1.1693971923001296, -2.1768136494850046, 0.09492718485511631, 0.8572613994662929, -2.39786527267812, -1.159171198248098, 1.0560736173734917, -0.25701076898467345, -1.0981986938209587, -0.3735956685207676, -0.5425883064344673, 0.7244473133352144, 0.4538858794596457], [-0.2802283393557666, -0.6653426856286421, -0.0538756638943387, 1.325155927345223, 0.3490261125780868, 0.6389455936428303, -0.1506815972085555, -1.2918162500762125, -0.7108435355871252, 0.6512365658972803, 0.10321610258141592, -0.9038398688268016, -1.617870425602848, 0.11068190035142705, -0.339531008698164, -0.350530218612727], [-0.3294499574610804, 0.28390511348310316, 0.29444772495280114, 0.9621285502204014, 0.9284215804413863, 1.3320930804934341, 0.8063630567820633, -0.3356080333276904, -0.05426894624359061, 0.4942150948314312, -0.31029227000142345, 0.5494387802798892, 1.1536094769949883, -0.8012460099660716, -1.770384050846499, 1.1936549552915572], [0.01775991033501458, 0.7086698231113021, -1.3323053543858547, 1.7329925604781644, 0.7386198894684642, -0.12607417369146828, -0.3519880388050872, 0.11389588045524304, -0.16673134056534972, -0.2786901272438552, -1.1195999755255746, 1.262809438921045, -0.039577684569559124, 0.07463563738960281, 0.09291598860047003, -2.358217336741871], [0.5271414581243252, -0.32915493062535894, -0.2892284645923461, 0.06978787257481246, -0.9724756053284209, 0.8433808099877073, -0.6958646238070096, -0.9036580547128793, -0.8621896009823745, 0.15875269513992676, 0.7457794743624686, -0.5950532355462611, -1.510186675915287, 1.062490693153603, -1.0650835950433684, 1.194830368529916], [-0.05429587697711575, -0.6983498940240875, -0.6400399491357172, 0.4618661736925272, 0.8893717949967992, -0.19817022758122477, 0.946934382371629, 2.08697820766597, -0.20830834526001082, -1.1494466412305686, -1.0544003485304514, -1.3903353550011621, -0.639665450940511, -0.7237792369318543, -1.3251033289662522, 0.5301842547135]], "dtype": "float64"}, {"__nd__": [-2.6113390049731797, -1.3207989716931114, 2.0152518061838767, 1.0378024284634166, 0.5227916833432323, 0.45725905895534386, 1.7709371728004815, 0.5356296362853685, 0.0775770872272132, -1.2301601607845671, -0.07366681732892769, -0.7338839983148214, -0.7061752749030227, -0.7142389333341079, -0.04464686469247794, -0.32971081974207533], "dtype": "float64"}, {"__nd__": [[0.8711105109344268, -1.3261814419117322, -0.16947019674286637, 1.8594974964345012, 0.0388973371949769], [1.198615221448642, 1.1368634972033143, 1.3166478203373606, 0.7381191695252466, -1.3804460312528912], [-0.2673877057870027, -0.38033558246862037, -1.5488553043334206, -0.4638816211751379, -1.2500798351337994], [-1.6655514460193832, 0.21089625656145172, -0.04359127244566091, -0.6878149095751852, -0.9505863626128569], [-0.541273442072544, -0.39409036937385966, -0.4081102825826052, -0.052707747928468524, -0.43554067396098334], [0.3869719823711992, 0.5228708948391112, 0.35786967764803124, -0.6631035145259176, -0.6165061466820784], [0.4348641034709542, -0.197042778476594, -0.04315170860457781, -2.4919952503112044, -2.700103979886236], [0.33714819206874613, 0.3295586563507561, 1.3934318714435818, 0.7211631188958838, 0.22173880016804176], [0.01636245608868217, -1.426740428209398, 0.8831758896325703, -0.6035860153728927, 0.5024844545833684], [0.20004164543806374, 0.272097020758701, 0.8926752155435728, -0.48986611197868063, -0.3872351816195507], [-0.8558614733519333, -0.37715202345150317, 1.6659407140512403, -0.3633186666203786, 1.1856163121936107], [0.8260114337421408, 0.7292817915893176, 0.8030151370721609, -1.0142765548969686, 0.37344767888543307], [1.588412097484749, -0.6957959363309963, -0.0471070855877405, -0.6891591571615298, 0.31701151121305016], [-1.0785301318658582, 1.6096797942508279, -0.37683478895943645, -1.542661572000243, 1.790790910814277], [0.970522894800899, 0.518511481410699, 0.5325875434742989, -0.3577980078127549, -0.22329559977220623], [0.7240192298296324, 0.9176249577098493, 0.9326124031440995, 0.2125509856107767, -0.466388093105908]], "dtype": "float64"}, {"__nd__": [-0.19208732917753973, 1.1095997115146592, 1.4399181149572342, -0.34131239008610353, -0.47730400254445376], "dtype": "float64"}], "e": {"__tup__": [{"__nd__": [[3.7485314897384643, -1.8625204174937975, 7.247114173503425, -9.709392015799109, -3.9577212451480768], [-0.8997743921974557, 6.992383829757271, 7.1814163206180295, -9.965357525537838, -11.230286540015786], [4.483965947819415, 0.17480181100008108, 10.09440473753747, -8.080819478653122, -12.401979020569547], [13.939286789685607, 2.5849431591397316, 4.655402249593357, -14.920340875083973, -19.900473193738303], [-2.0727303762341664, 1.983852370910568, 2.9139043900596757, -7.680258416111972, -11.787899324447933], [5.693915493114058, 3.925480106890668, 5.544541066430581, -20.912242066318615, -24.927191645816475], [6.681542410199491, -1.9940680203389476, 12.268507945625405, -22.251909524320563, -26.77314578485764], [6.655573779330075, 1.5682288439494292, -0.0971875699191076, -9.476053262323623, -9.964435027525623], [-1.802960107561987, 0.5050131951970083, -4.130365081958221, -10.273712516242764, -15.767700453737435], [-0.9788782711886732, 3.5818767014088735, 6.066174283762604, -3.2392970952271454, -11.78976525988937], [4.346912164741828, 3.0167328205784765, 11.37479689172602, -5.855298659846086, -11.83080845098123], [-4.775841763591126, 4.362652588247027, 2.5094377097702782, -7.409736337849346, -7.383496164278353]], "dtype": "float64"}, {"__nd__": [2, 2, 2, 0, 2, 0, 2, 0, 1, 2, 2, 1], "dtype": "int64"}, 229]}}]'
import json, types
harness = types.ModuleType('harness'); exec(_HARNESS, harness.__dict__)
res = harness.grade(mlp_predict, json.loads(_TESTS), 'torch', *[1e-06, 1e-08])
for i, r in enumerate(res): print(f"test {i+1}: {'pass' if r['ok'] else 'FAIL  ' + r['why']}")
print(f"{sum(r['ok'] for r in res)}/{len(res)} passed")